# Batch Product Evidence Harness

Runs a CSV/XLSX batch and highlights the **production-grade product URL handoff**.

For browser/scraping/coding team handoff, use only rows where:

```text
production_url_ready == True
production_url_status == "PRODUCTION_READY_EXACT_SCRAPABLE_BROWSER_URL"
needs_review == False
```

Rows that fail this gate can still have `product_url`, but they are review-only.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT

In [ ]:
input_path = PROJECT_ROOT / "data" / "products.csv"
output_path = PROJECT_ROOT / "outputs" / "final_submission.csv"
workers = 4

print("Input:", input_path)
print("Output:", output_path)

In [ ]:
# Preview input while preserving EAN/GTIN as text.
# Required columns: row_id, main_text, country_code
# Optional columns: ean, retailer_name, language_code, region
if input_path.exists():
    display(pd.read_csv(input_path, dtype=str).head())
else:
    print("Input file not found. Update input_path above.")

In [ ]:
cmd = [
    sys.executable,
    "batch_main.py",
    "--input", str(input_path),
    "--output", str(output_path),
    "--workers", str(workers),
]
print(" ".join(cmd))

# Uncomment to run from notebook. For long batches, terminal execution is easier to monitor.
# subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

In [ ]:
if output_path.exists():
    df = pd.read_csv(output_path, dtype=str)
    print("Rows:", len(df))
    display(df.head())
else:
    df = pd.DataFrame()
    print("Run batch_main.py first to create:", output_path)

In [ ]:
handoff_cols = [
    "row_id", "main_text", "country_code", "retailer_name",
    "product_url", "production_url_ready", "production_url_status",
    "browser_openable", "highly_scrapable", "exact_product_url_match",
    "needs_review", "verified_exact_url", "quality_tier",
    "coding_readiness_status", "failure_taxonomy",
    "production_url_reasons", "product_coding_input_path",
]

if not df.empty:
    display(df[[c for c in handoff_cols if c in df.columns]].head(25))

In [ ]:
if not df.empty:
    ready = df[
        (df.get("production_url_ready", "").astype(str).str.lower().isin(["true", "1", "yes"]))
        & (df.get("production_url_status", "") == "PRODUCTION_READY_EXACT_SCRAPABLE_BROWSER_URL")
        & (~df.get("needs_review", "true").astype(str).str.lower().isin(["true", "1", "yes"]))
    ].copy()

    print("Production-ready handoff rows:", len(ready), "/", len(df))
    display(ready[[c for c in handoff_cols if c in ready.columns]].head(50))
else:
    ready = pd.DataFrame()

In [ ]:
if not df.empty:
    review_only = df[~df.index.isin(ready.index)].copy()
    print("Review-only rows:", len(review_only))
    display(review_only[[c for c in handoff_cols if c in review_only.columns]].head(50))

In [ ]:
metrics_path = PROJECT_ROOT / "outputs" / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
else:
    print("metrics.json not found:", metrics_path)

In [ ]:
summary_path = PROJECT_ROOT / "outputs" / "batch_summary.md"
if summary_path.exists():
    print(summary_path.read_text(encoding="utf-8")[:6000])
else:
    print("batch_summary.md not found:", summary_path)

In [ ]:
# Inspect one row artifact packet.
if not df.empty:
    row_id = df.iloc[0]["row_id"]
    row_dir = PROJECT_ROOT / "output" / str(row_id)
    print("Row artifact directory:", row_dir)
    if row_dir.exists():
        for p in sorted(row_dir.iterdir()):
            print("-", p.name)

    coding_path = row_dir / "product_coding_input.json"
    if coding_path.exists():
        payload = json.loads(coding_path.read_text(encoding="utf-8"))
        print(json.dumps({
            "selected_url": payload.get("selected_url"),
            "verified_exact_url": payload.get("verified_exact_url"),
            "quality_tier": payload.get("quality_tier"),
            "coding_readiness_status": payload.get("coding_readiness_status"),
            "review_flags": payload.get("review_flags"),
        }, indent=2, ensure_ascii=False))